In [13]:
import pandas as pd
import re
import numpy as np

In [14]:
from pathlib import Path
import pandas as pd

RACINE = Path.cwd().parent            # notebook dans notebooks/ -> racine du projet
df = pd.read_csv(RACINE / "data" / "raw" / "yield_curve.csv")


In [15]:
df = df.set_index('DATE')

In [16]:
df

,TIME PERIOD,AAA yield curve - 10-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10M),AAA yield curve - 10-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y),AAA yield curve - 10-year 11-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y11M),AAA yield curve - 10-year 1-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y1M),AAA yield curve - 10-year 3-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y3M),AAA yield curve - 10-year 4-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y4M),AAA yield curve - 10-year 5-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y5M),AAA yield curve - 10-year 6-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y6M),AAA yield curve - 10-year 7-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y7M),...,AAA yield curve - 9-year 1-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y1M),AAA yield curve - 9-year 2-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y2M),AAA yield curve - 9-year 4-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y4M),AAA yield curve - 9-year 5-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y5M),AAA yield curve - 9-year 6-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y6M),AAA yield curve - 9-year 8-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y8M),AAA yield curve - 9-year 9-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y9M),AAA yield curve - spread between 10-year and 1-year spot rates (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SRS_10Y_1Y),AAA yield curve - spread between 10-year and 2-year spot rates (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SRS_10Y_2Y),AAA yield curve - spread between 10-year and 3-month spot rates (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SRS_10Y_3M)
DATE,,,,,,,,,,,,,,,,,,,,,
2004-09-06,06 Sep 2004,2.239924,4.209220,4.295947,4.217575,4.233993,4.242057,4.250027,4.257905,4.265691,...,4.110371,4.119913,4.138649,4.147847,4.156935,4.174783,4.183548,1.910381,1.567958,2.175048
2004-09-07,07 Sep 2004,2.266495,4.209626,4.294336,4.217781,4.233808,4.241682,4.249466,4.257161,4.264767,...,4.113248,4.122543,4.140801,4.149766,4.158625,4.176030,4.184578,1.880735,1.529860,2.168733
2004-09-08,08 Sep 2004,2.281809,4.228419,4.311649,4.236434,4.252185,4.259923,4.267571,4.275131,4.282604,...,4.133646,4.142790,4.160749,4.169566,4.178278,4.195392,4.203796,1.881753,1.521877,2.184035
2004-09-09,09 Sep 2004,2.249641,4.161872,4.247733,4.170128,4.186358,4.194336,4.202224,4.210024,4.217735,...,4.064465,4.073847,4.092282,4.101339,4.110291,4.127884,4.136529,1.852884,1.515648,2.124761
2004-09-10,10 Sep 2004,2.217962,4.120981,4.208462,4.129396,4.145937,4.154066,4.162103,4.170050,4.177907,...,4.021659,4.031228,4.050031,4.059267,4.068395,4.086334,4.095147,1.849416,1.531327,2.086336
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-22,22 Oct 2026,2.826585,3.384401,3.426277,3.388340,3.396141,3.400002,3.403838,3.407646,3.411428,...,3.339542,3.343726,3.352036,3.356160,3.360263,3.368402,3.372437,0.514667,0.376906,0.796822
2026-10-23,23 Oct 2026,2.812834,3.369406,3.412452,3.373454,3.381471,3.385440,3.389382,3.393297,3.397184,...,3.323307,3.327607,3.336147,3.340385,3.344601,3.352965,3.357112,0.515880,0.385945,0.782718
2026-10-26,26 Oct 2026,2.835468,3.373550,3.415019,3.377446,3.385164,3.388987,3.392784,3.396556,3.400302,...,3.329281,3.333402,3.341591,3.345658,3.349706,3.357741,3.361726,0.490933,0.348170,0.806282


In [17]:
#extraction et groupage des maturités
def mat(c):
    m = re.search(r"\.SR_(.*)\)$", c) #on regarde la structure qui se termoiner par .SR_ suivi de car et se terminant par )
    if not m:
        return None
    a = re.search(r"(\d+)Y", m.group(1)) #on cherche un nombre suivi de Y
    b = re.search(r"(\d+)M", m.group(1)) #on cherche un nombre suivi de M
    
    return (int(a.group(1)) if a else 0) + (int(b.group(1)) if b else 0) / 12 # on convertir en nombre 

T = pd.Series({c: mat(c) for c in df.columns}).dropna()

courbe = (df[T.index]                     
          .rename(columns=T.to_dict())     
          .sort_index(axis=1))             # trie de la plus courte à la plus longue
print(pd.cut(T, [0, 2, 5, 10, 20, 30]).value_counts().sort_index()) # réparit chaque maturtié de T dans l'intervalle

(0, 2]      14
(2, 5]      16
(5, 10]     31
(10, 20]    68
(20, 30]    68
Name: count, dtype: int64


In [18]:
courbe

,0.250000,0.333333,0.416667,0.500000,0.583333,0.666667,0.750000,0.833333,0.916667,1.000000,...,28.750000,29.000000,29.166667,29.333333,29.416667,29.500000,29.750000,29.833333,29.916667,30.000000
DATE,,,,,,,,,,,,,,,,,,,,,
2004-09-06,2.034172,2.063283,2.092544,2.121920,2.151377,2.180882,2.210407,2.239924,2.269409,2.298838,...,4.970365,4.974153,4.976643,4.979105,4.980325,4.981539,4.985139,4.986326,4.987506,4.988680
2004-09-07,2.040893,2.073717,2.106364,2.138822,2.171075,2.203112,2.234922,2.266495,2.297820,2.328891,...,4.957418,4.961158,4.963615,4.966045,4.967249,4.968447,4.972000,4.973172,4.974336,4.975495
2004-09-08,2.044384,2.079252,2.113815,2.148064,2.181994,2.215598,2.248871,2.281809,2.314408,2.346666,...,4.961220,4.964876,4.967279,4.969654,4.970832,4.972003,4.975477,4.976623,4.977762,4.978894
2004-09-09,2.037111,2.067808,2.098422,2.128935,2.159329,2.189588,2.219696,2.249641,2.279409,2.308988,...,4.927853,4.931719,4.934260,4.936773,4.938018,4.939257,4.942932,4.944143,4.945347,4.946545
2004-09-10,2.034645,2.060137,2.085928,2.111976,2.138239,2.164681,2.191267,2.217962,2.244738,2.271566,...,4.899559,4.903483,4.906062,4.908612,4.909876,4.911133,4.914862,4.916092,4.917314,4.918530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-22,2.587579,2.632406,2.673119,2.710076,2.743610,2.774024,2.801597,2.826585,2.849226,2.869734,...,3.631540,3.628918,3.627126,3.625298,3.624372,3.623436,3.620581,3.619612,3.618636,3.617651
2026-10-23,2.586687,2.629188,2.667756,2.702740,2.734458,2.763204,2.789247,2.812834,2.834191,2.853526,...,3.633012,3.630523,3.628818,3.627077,3.626193,3.625301,3.622572,3.621645,3.620710,3.619767
2026-10-26,2.567269,2.618170,2.664210,2.705822,2.743404,2.777321,2.807907,2.835468,2.860286,2.882618,...,3.616727,3.614033,3.612192,3.610316,3.609365,3.608405,3.605476,3.604483,3.603482,3.602472


In [27]:
X= pd.Series(courbe.columns)

In [28]:
X

0       0.250000
1       0.333333
2       0.416667
3       0.500000
4       0.583333
         ...    
192    29.500000
193    29.750000
194    29.833333
195    29.916667
196    30.000000
Length: 197, dtype: float64

In [19]:
T

AAA yield curve - 10-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10M)                0.833333
AAA yield curve - 10-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y)                10.000000
AAA yield curve - 10-year 11-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y11M)    10.916667
AAA yield curve - 10-year 1-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y1M)      10.083333
AAA yield curve - 10-year 3-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y3M)      10.250000
                                                                                           ...    
AAA yield curve - 9-year 4-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y4M)         9.333333
AAA yield curve - 9-year 5-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y5M)         9.416667
AAA yield curve - 9-year 6-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y6M)         9.500000
AAA yield curve - 9-year 8-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_9Y8M)         9.666667
AAA yield 

In [20]:
courbe

,0.250000,0.333333,0.416667,0.500000,0.583333,0.666667,0.750000,0.833333,0.916667,1.000000,...,28.750000,29.000000,29.166667,29.333333,29.416667,29.500000,29.750000,29.833333,29.916667,30.000000
DATE,,,,,,,,,,,,,,,,,,,,,
2004-09-06,2.034172,2.063283,2.092544,2.121920,2.151377,2.180882,2.210407,2.239924,2.269409,2.298838,...,4.970365,4.974153,4.976643,4.979105,4.980325,4.981539,4.985139,4.986326,4.987506,4.988680
2004-09-07,2.040893,2.073717,2.106364,2.138822,2.171075,2.203112,2.234922,2.266495,2.297820,2.328891,...,4.957418,4.961158,4.963615,4.966045,4.967249,4.968447,4.972000,4.973172,4.974336,4.975495
2004-09-08,2.044384,2.079252,2.113815,2.148064,2.181994,2.215598,2.248871,2.281809,2.314408,2.346666,...,4.961220,4.964876,4.967279,4.969654,4.970832,4.972003,4.975477,4.976623,4.977762,4.978894
2004-09-09,2.037111,2.067808,2.098422,2.128935,2.159329,2.189588,2.219696,2.249641,2.279409,2.308988,...,4.927853,4.931719,4.934260,4.936773,4.938018,4.939257,4.942932,4.944143,4.945347,4.946545
2004-09-10,2.034645,2.060137,2.085928,2.111976,2.138239,2.164681,2.191267,2.217962,2.244738,2.271566,...,4.899559,4.903483,4.906062,4.908612,4.909876,4.911133,4.914862,4.916092,4.917314,4.918530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-22,2.587579,2.632406,2.673119,2.710076,2.743610,2.774024,2.801597,2.826585,2.849226,2.869734,...,3.631540,3.628918,3.627126,3.625298,3.624372,3.623436,3.620581,3.619612,3.618636,3.617651
2026-10-23,2.586687,2.629188,2.667756,2.702740,2.734458,2.763204,2.789247,2.812834,2.834191,2.853526,...,3.633012,3.630523,3.628818,3.627077,3.626193,3.625301,3.622572,3.621645,3.620710,3.619767
2026-10-26,2.567269,2.618170,2.664210,2.705822,2.743404,2.777321,2.807907,2.835468,2.860286,2.882618,...,3.616727,3.614033,3.612192,3.610316,3.609365,3.608405,3.605476,3.604483,3.603482,3.602472


In [21]:
#calcul des stat sur la densité des points de données par tranche de maturité

bins = [0, 2, 5, 10, 20, 30]
effectifs = pd.cut(T, bins).value_counts().sort_index()
largeurs = pd.Series(np.diff(bins), index=effectifs.index)
print(pd.DataFrame({
    "points": effectifs,
    "largeur": largeurs,
    "points_par_an": (effectifs / largeurs).round(1),
}))

#tire aléatoirement un échantillon de colonne à masquer poiur le test
def choisir_caches(T, bins, n_parts=3,seed=42):
    rng = np.random.default_rng(seed)
    T_int = T[(T > T.min()) & (T < T.max())]   # extrémités exclues = ancrage
    caches = [] 
   
    for lo, hi in zip(bins[:-1], bins[1:]): # une itération par tranche : (0,2], (2,5], (5,10], (10,20], (20,30]
        interv=np.linspace(lo, hi, n_parts + 1) # on sépare l'interval en 3 parts pour avoir des valeurs de test équilibrés
        
        for lo_part, hi_part in zip(interv[:-1], interv[1:]): 
            dispo = T_int[(T_int > lo_part) & (T_int <= hi_part)].index
            if len(dispo)==0:  #on regarde si dans la part il y a au moins une maturité
                raise ValueError(f"Aucune maturité dans ]{lo_part:.2f}, {hi_part:.2f}]") 
            else :
                caches += list(rng.choice(dispo, size=1, replace=False)) #on tire une col sans remise dans chaque part de l'intervalle
    
    return caches

caches = choisir_caches(T, bins)
ancrage = [c for c in T.index if c not in caches]

          points  largeur  points_par_an
(0, 2]        14        2            7.0
(2, 5]        16        3            5.3
(5, 10]       31        5            6.2
(10, 20]      68       10            6.8
(20, 30]      68       10            6.8


In [22]:
ancrage

['AAA yield curve - 10-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10M)',
 'AAA yield curve - 10-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y)',
 'AAA yield curve - 10-year 11-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y11M)',
 'AAA yield curve - 10-year 1-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y1M)',
 'AAA yield curve - 10-year 4-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y4M)',
 'AAA yield curve - 10-year 5-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y5M)',
 'AAA yield curve - 10-year 6-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y6M)',
 'AAA yield curve - 10-year 7-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y7M)',
 'AAA yield curve - 10-year 9-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y9M)',
 'AAA yield curve - 11-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_11M)',
 'AAA yield curve - 11-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_11Y)',
 'AAA yield curve - 11-year 10-month spot rate (YC.B.U2.EUR.4F.G_N_A

In [23]:
caches

['AAA yield curve - 4-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_4M)',
 'AAA yield curve - 1-year 4-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y4M)',
 'AAA yield curve - 2-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y)',
 'AAA yield curve - 2-year 8-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y8M)',
 'AAA yield curve - 3-year 5-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_3Y5M)',
 'AAA yield curve - 5-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y)',
 'AAA yield curve - 5-year 4-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y4M)',
 'AAA yield curve - 7-year 8-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_7Y8M)',
 'AAA yield curve - 8-year 6-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_8Y6M)',
 'AAA yield curve - 10-year 3-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y3M)',
 'AAA yield curve - 15-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_15Y)',
 'AAA yield curve - 20-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_20Y)',
 'AAA yield